# Colab Inference Setup via Git + uv
This notebook provides a frictionless setup for running inference on Google Colab.
It uses `uv` for fast dependency management and execution, and Git to pull your latest code.

In [ ]:
# 1. Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

In [ ]:
# 2. Clone Repository with Authentication
# Add a GitHub Personal Access Token (classic) to Colab Secrets as 'GITHUB_TOKEN'
# Replace with your actual GitHub username
GITHUB_USERNAME = "arnavgarg"
REPO_NAME = "llm"

import os
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print("GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

if GITHUB_TOKEN:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
else:
    REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone $REPO_URL

%cd $REPO_NAME

In [ ]:
# 3. Install Dependencies System-wide for Colab Environment
!uv pip install --system -e .

In [ ]:
# 4. Weights & Biases Authentication
# For a frictionless experience, add your W&B API Key to Colab Secrets as 'WANDB_API_KEY'.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print("W&B API Key loaded from Colab Secrets.")
except Exception as e:
    print("WANDB_API_KEY not found in Colab Secrets. You may be prompted to login during execution.")

In [ ]:
# 5. Update Code (Run this cell when you push new local changes)
!git pull
!uv pip install --system -e .

## 6. Inference Setup
Import necessary modules and fetch the run configuration from W&B.

In [ ]:
import torch
import wandb

# Ensure you have your project structured properly for imports
from models.gpt import GPT
from tokenizers.character import CharacterTokenizer
from dataloaders.tiny_shakespeare import _download
from inference.generator import TextGenerator

# Set device
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 7. Download Model Weights and Config from W&B
# Initialize API
api = wandb.Api()

# Specify your W&B entity, project, and run ID
ENTITY = "your_wandb_username"  # Replace with your W&B username or team name
PROJECT = "llm"
RUN_ID = "YOUR_RUN_ID"          # Replace with the actual run ID you want to load

run_path = f"{ENTITY}/{PROJECT}/{RUN_ID}"
run = api.run(run_path)
config = run.config
print(f"Loaded config for run {RUN_ID}")

# Find the best/latest model file (assuming they are named 'weights/model_epoch_*.pt')
# You can also hardcode the exact filename if preferred.
model_files = [f for f in run.files() if f.name.startswith("weights/model_epoch_")]
if not model_files:
    raise ValueError("No model weights found in this run!")

# Sort to get the latest epoch
model_files.sort(key=lambda x: int(x.name.split("_")[-1].split(".")[0]))
best_model_file = model_files[-1]
print(f"Downloading {best_model_file.name}...")
best_model_file.download(replace=True)
model_path = best_model_file.name

In [ ]:
# 8. Reconstruct Tokenizer
print("Reconstructing Tokenizer Vocabulary...")
text = _download()
tokenizer = CharacterTokenizer()
tokenizer.fit(text)
print(f"Vocabulary size: {tokenizer.vocab_size}")

In [ ]:
# 9. Initialize Model
model = GPT(
    vocab_size=tokenizer.vocab_size,
    d_context=config.get("context_length", 512),
    d_model=config.get("d_model", 32),
    num_heads=config.get("num_heads", 2),
    d_ff=config.get("d_ff", 128),
    num_layers=config.get("depth", 2)
)

# Load weights
print(f"Loading weights from {model_path}...")
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
print("Model loaded successfully!")

In [ ]:
# 10. Generate Text
generator = TextGenerator(model, tokenizer, context_length=config.get("context_length", 512), device=device)

prompt = "ROMEO:\n"
print("Prompt:")
print(prompt)
print("-" * 20)
print("Generated Continuation:")

# Generate 200 tokens
output = generator.generate(prompt, max_new_tokens=200, temperature=0.8)
print(output)